# Heatmap Localization Visualizations

In [ ]:
def visualize_heatmap(model, dataset, device, idx=0):
    model.eval()

    x, y, fname = dataset[idx]

    with torch.inference_mode():
        pred = torch.sigmoid(model(x.unsqueeze(0).to(device)))[0, 0].cpu().numpy()

    img = x.cpu().permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    gt = y[0].cpu().numpy()

    image_path = Path("./evaluation/train") / fname
    raw_img = np.array(Image.open(image_path).convert("RGB"))
    fig, axs = plt.subplots(1, 3, figsize=(24, 7))

    axs[0].imshow(raw_img)
    axs[0].set_title("Input Image", fontsize=16)
    axs[0].axis("off")

    axs[1].imshow(gt, cmap="magma", vmin=0, vmax=1)
    axs[1].set_title("Ground Truth", fontsize=16)
    axs[1].axis("off")

    axs[2].imshow(pred, cmap="magma", vmin=0, vmax=1)
    axs[2].set_title("Prediction", fontsize=16)
    axs[2].axis("off")

    plt.tight_layout()
    plt.show()


import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
import cv2
import scipy.ndimage as ndi
from skimage.segmentation import watershed


def visualize_sample(model, dataset, device, idx=0, show_steps=True):
    model.eval()

    x, y, fname = dataset[idx]

    with torch.inference_mode():
        pred = torch.sigmoid(model(x.unsqueeze(0).to(device)))[0, 0].cpu().numpy()

    img = x.cpu().permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    image_path = Path("./evaluation/train") / fname

    centroids, num_cells = extract_cell_coordinates(
        pred, threshold=0.15, min_distance=8
    )
    centroids_merged = merge_close_centroids(centroids, distance_threshold=15)

    img_file = pyvips.Image.new_from_file(image_path, access="sequential")

    patch = np.ndarray(
        buffer=img_file.write_to_memory(),
        dtype=np.uint8,
        shape=[img_file.height, img_file.width, img_file.bands],
    )
    patch = cv2.GaussianBlur(patch, (3, 3), 0)
    masked = binarize_HSV(patch)

    dist = ndi.distance_transform_edt(masked)

    markers = np.zeros(masked.shape, dtype=np.int32)
    for i, (yy, xx) in enumerate(centroids_merged, start=1):
        markers[int(yy), int(xx)] = i

    labels = np.array(watershed(-dist, markers, mask=masked))
    labels, segmented_cells = filter_size(labels, len(centroids_merged), threshold=1000)
    bounding_boxes = calculate_bb(segmented_cells)

    final_bb = []
    for bb in bounding_boxes:
        final_bb.append(bb)

    if show_steps:
        fig, axs = plt.subplots(
            2,
            3,
        )

        axs[0, 0].imshow(np.array(Image.open(image_path).convert("RGB")))
        axs[0, 0].set_title("Original image")
        axs[0, 0].axis("off")

        axs[0, 1].imshow(pred, cmap="magma")
        axs[0, 1].set_title("Predicted heatmap")
        axs[0, 1].axis("off")

        axs[0, 2].imshow(masked, cmap="gray")
        axs[0, 2].set_title("Binarized Image")
        axs[0, 2].axis("off")

        axs[1, 0].imshow(dist, cmap="magma")
        axs[1, 0].set_title("Distance transform")
        axs[1, 0].axis("off")

        (x, y) = np.nonzero(markers)
        axs[1, 1].imshow(masked, cmap="gray")
        axs[1, 1].scatter(y, x)

        axs[1, 1].set_title("Markers / centroids")
        axs[1, 1].axis("off")

        axs[1, 2].imshow(np.array(Image.open(image_path).convert("RGB")))
        axs[1, 2].imshow(labels, cmap="gist_ncar", alpha=0.45)
        axs[1, 2].set_title("Watershed-separated cells")
        axs[1, 2].axis("off")

        plt.tight_layout()
        plt.show()

    return final_bb, centroids_merged, labels

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.utils import *
import albumentations as A
import pyvips
from torch.utils.data import Dataset, DataLoader, random_split
from albumentations.pytorch import ToTensorV2
from torchvision import transforms
import torch
from torch.utils.data import Subset
import torch.nn as nn
import segmentation_models_pytorch as smp
from tqdm import tqdm
import gc
from PIL import Image
from sklearn.cluster import DBSCAN


class MicrogliaHeatmapDataset(Dataset):

    def __init__(self, image_dir, json_path, sigma=20.0, augment=False):
        self.image_dir = Path(image_dir)
        self.annotations = extract_points_from_json(json_path)
        self.file_names = [f.name for f in self.image_dir.glob("*.png")]
        self.sigma = sigma
        self.augment = augment
        self.train_aug = A.Compose(
            [
                A.Normalize(),
                ToTensorV2(),
            ]
        )
        self.val_aug = A.Compose(
            [
                A.Normalize(),
                ToTensorV2(),
            ]
        )
        self.heatmaps = {}
        for fname in self.file_names:
            with Image.open(self.image_dir / fname) as img:
                w, h = img.size
            pts = self.annotations.get(fname, [])
            self.heatmaps[fname] = make_gaussian_heatmap(
                (h, w), pts, sigma=self.sigma
            ).astype(np.float32)
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        with Image.open(self.image_dir / fname) as img:
            img = np.array(img.convert("RGB"))

        aug = self.train_aug if self.augment else self.val_aug
        result = aug(image=img, mask=self.heatmaps[fname])
        x = result["image"].float()
        y = result["mask"].unsqueeze(0).float()
        return x, y, fname


dataset_split = "train"
IMAGE_DIR = f"./evaluation/{dataset_split}"
JSON_PATH = "./evaluation/annotations/gt_points.json"
SAVE_PATH = "./checkpoints/resnet18_sigma15_aug.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
NUM_EPOCHS = 30
LR = 1e-4
VAL_SPLIT = 0.2
SIGMA = 15.0

train_dataset = MicrogliaHeatmapDataset(IMAGE_DIR, JSON_PATH, sigma=SIGMA, augment=True)
full_dataset = MicrogliaHeatmapDataset(IMAGE_DIR, JSON_PATH, sigma=SIGMA, augment=False)

val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
generator = torch.Generator().manual_seed(30)
train_indices, val_indices = random_split(
    range(len(full_dataset)), [train_size, val_size], generator=generator
)


train_dataset = Subset(train_dataset, train_indices.indices)
val_dataset = Subset(full_dataset, val_indices.indices)
checkpoint = torch.load("./checkpoints/resnet18_sigma15_aug.pth", map_location=DEVICE)
model = smp.Unet().to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


from scipy.ndimage import maximum_filter, label, center_of_mass
import numpy as np


def extract_cell_coordinates(heatmap, threshold=0.7, min_distance=20, border=10):
    """
    Given a heatmap, extract potential cell coordinates by finding local maxima above a threshold.
    border: number of pixels to ignore around the image edges
    """
    local_max = heatmap == maximum_filter(
        heatmap, size=min_distance, mode="constant", cval=-np.inf
    )
    local_max &= heatmap > threshold

    # remove edge detections
    if border > 0:
        local_max[:border, :] = False
        local_max[-border:, :] = False
        local_max[:, :border] = False
        local_max[:, -border:] = False

    labeled, num_cells = label(local_max)
    centroids = center_of_mass(heatmap, labeled, range(1, num_cells + 1))

    # remove any invalid values
    centroids = [(y, x) for y, x in centroids if not np.isnan(y) and not np.isnan(x)]

    return centroids, len(centroids)


def merge_close_centroids(centroids, distance_threshold=30):
    """Given potential cell centroids, merge those that are within the specified distance threshold into a single centroid using mean location"""
    if len(centroids) == 0:
        return []

    pts = np.array(centroids)
    labels = DBSCAN(eps=distance_threshold, min_samples=1).fit_predict(pts)

    merged = []
    for lab in np.unique(labels):
        cluster_pts = pts[labels == lab]
        merged_centroid = cluster_pts.mean(axis=0)
        merged.append(tuple(merged_centroid))

    return merged


for i in range(20, 40):
    visualize_sample(
        model,
        train_dataset,
        DEVICE,
        idx=i,
    )